# Train encoders — 6 emotions

Trains audio and video emotion classifiers on **happy**, **sad**, **angry**, **fearful**, **disgust**, **surprised** (RAVDESS `emotion_idx` 2, 3, 4, 5, 6, 7). Expanded experiment grid vs. 3-emotion version.

In [1]:
!pip install -q transformers wandb

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: katrinpochtar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import gc
import json
import random
import warnings
from functools import partial
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchaudio
from sklearn.metrics import accuracy_score, f1_score
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    Wav2Vec2ForSequenceClassification,
    HubertForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    AutoImageProcessor,
    TimesformerForVideoClassification,
)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
OUT_DIR = Path("/content/trained_encoders_6emotions")
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_WANDB = True
WANDB_PROJECT = "uncanny-valley-encoders-6emo"

# Same ordering as 01_data_preprocessing.ipynb EMOTION_TO_IDX
EMOTION_NAME_TO_IDX = {
    "neutral": 0, "calm": 1, "happy": 2, "sad": 3,
    "angry": 4, "fearful": 5, "disgust": 6, "surprised": 7,
}
EMOTIONS = ["happy", "sad", "angry", "fearful", "disgust", "surprised"]
ALLOWED_EMOTION_IDX = {EMOTION_NAME_TO_IDX[e] for e in EMOTIONS}
REMAP = {EMOTION_NAME_TO_IDX[e]: i for i, e in enumerate(EMOTIONS)}
NUM_EMOTIONS = len(EMOTIONS)
DEFAULT_LABEL_SMOOTHING = 0.1
DEFAULT_WEIGHT_DECAY = 0.01

print(f"Device: {DEVICE}")
print(f"Classes: {NUM_EMOTIONS}, Emotions: {EMOTIONS}")
print(f"RAVDESS emotion_idx: {sorted(ALLOWED_EMOTION_IDX)} -> labels 0..{NUM_EMOTIONS-1}")

Device: cuda
Classes: 6, Emotions: ['happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
RAVDESS emotion_idx: [2, 3, 4, 5, 6, 7] -> labels 0..5


In [4]:
class EmotionDataset(Dataset):
    def __init__(self, metadata_path, split, modality):
        with open(metadata_path) as f:
            data = json.load(f)
        self.samples = [
            s for s in data
            if s["split"] == split and s["emotion_idx"] in ALLOWED_EMOTION_IDX
        ]
        self.modality = modality

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        item = {"emotion": REMAP[s["emotion_idx"]]}
        if self.modality == "audio":
            wav, _ = torchaudio.load(s["audio_path"])
            item["audio"] = wav.squeeze(0)
        else:
            frames = np.load(s["frames_path"])
            item["video"] = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        return item


def collate_fn(batch):
    out = {"emotion": torch.tensor([b["emotion"] for b in batch])}
    if "audio" in batch[0]:
        out["audio"] = [b["audio"] for b in batch]
    if "video" in batch[0]:
        out["video"] = torch.stack([b["video"] for b in batch])
    return out


def class_counts(metadata_path, split):
    with open(metadata_path) as f:
        data = json.load(f)
    counts = np.zeros(NUM_EMOTIONS, dtype=np.int64)
    for s in data:
        if s["split"] == split and s["emotion_idx"] in ALLOWED_EMOTION_IDX:
            counts[REMAP[s["emotion_idx"]]] += 1
    return counts


print("Train class counts:", dict(zip(EMOTIONS, class_counts(METADATA, "train").tolist())))
print("Val   class counts:", dict(zip(EMOTIONS, class_counts(METADATA, "val").tolist())))

Train class counts: {'happy': 136, 'sad': 136, 'angry': 136, 'fearful': 136, 'disgust': 136, 'surprised': 136}
Val   class counts: {'happy': 24, 'sad': 24, 'angry': 24, 'fearful': 24, 'disgust': 24, 'surprised': 24}


In [5]:
def crop_audio(wav, sr, duration, train):
    L = int(round(duration * sr))
    n = wav.numel()
    if n <= L:
        return torch.nn.functional.pad(wav, (0, L - n))
    start = torch.randint(0, n - L + 1, ()).item() if train else (n - L) // 2
    return wav[start:start + L]


def crop_video(video, n_frames, train):
    T = video.shape[0]
    if T <= n_frames:
        idx = torch.linspace(0, T - 1, n_frames).round().long()
        return video[idx]
    start = torch.randint(0, T - n_frames + 1, ()).item() if train else (T - n_frames) // 2
    return video[start:start + n_frames]


def prepare_audio(batch, processor, window_s, device, train=True):
    sr = 16000
    wavs = [crop_audio(a, sr, window_s, train).numpy() for a in batch["audio"]]
    enc = processor(wavs, sampling_rate=sr, return_tensors="pt", padding=True,
                    truncation=True, max_length=int(window_s * sr))
    kwargs = {"input_values": enc["input_values"].to(device)}
    if "attention_mask" in enc:
        kwargs["attention_mask"] = enc["attention_mask"].to(device)
    return kwargs, batch["emotion"].to(device)


def prepare_video(batch, processor, n_frames, device, train=True):
    clips = []
    for v in batch["video"]:
        clip = crop_video(v, n_frames, train)
        clips.append([clip[i].permute(1, 2, 0).numpy() for i in range(clip.shape[0])])
    enc = processor(clips, return_tensors="pt", do_rescale=False)
    return {"pixel_values": enc["pixel_values"].to(device)}, batch["emotion"].to(device)

In [6]:
def train_one_epoch(model, loader, prep_fn, optimizer, scaler, loss_fn):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    for batch in tqdm(loader, leave=False):
        kwargs, y = prep_fn(batch, train=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(**kwargs).logits
            loss = loss_fn(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        labels.extend(y.cpu().tolist())
    return {
        "loss": total_loss / len(loader),
        "acc": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


@torch.no_grad()
def evaluate(model, loader, prep_fn, loss_fn):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    for batch in tqdm(loader, leave=False):
        kwargs, y = prep_fn(batch, train=False)
        with autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(**kwargs).logits
            loss = loss_fn(logits, y)
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().tolist())
        labels.extend(y.cpu().tolist())
    return {
        "loss": total_loss / len(loader),
        "acc": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [7]:
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_loss_fn(cfg):
    ls = cfg.get("label_smoothing", DEFAULT_LABEL_SMOOTHING)
    weight = None
    if cfg.get("class_balanced", False):
        counts = class_counts(METADATA, "train").astype(np.float32)
        inv = counts.sum() / (NUM_EMOTIONS * np.clip(counts, 1, None))
        weight = torch.tensor(inv, dtype=torch.float32, device=DEVICE)
    return nn.CrossEntropyLoss(label_smoothing=ls, weight=weight)


def build_model_and_prep(cfg):
    modality = cfg["modality"]
    if modality == "audio":
        model_cls = (HubertForSequenceClassification if "hubert" in cfg["model"].lower()
                     else Wav2Vec2ForSequenceClassification)
        model = model_cls.from_pretrained(
            cfg["model"], num_labels=NUM_EMOTIONS, ignore_mismatched_sizes=True)
        processor = Wav2Vec2FeatureExtractor.from_pretrained(cfg["model"])
        prep_fn = partial(prepare_audio, processor=processor,
                          window_s=cfg.get("window_s", 3.0), device=DEVICE)
        if hasattr(model, "freeze_feature_encoder"):
            model.freeze_feature_encoder()
    else:
        model = TimesformerForVideoClassification.from_pretrained(
            cfg["model"], num_labels=NUM_EMOTIONS, ignore_mismatched_sizes=True)
        processor = AutoImageProcessor.from_pretrained(cfg["model"])
        prep_fn = partial(prepare_video, processor=processor,
                          n_frames=cfg.get("n_frames", 8), device=DEVICE)
        for n, p in model.named_parameters():
            if "classifier" not in n:
                p.requires_grad = False
    return model, processor, prep_fn


def run_experiment(cfg):
    seed_all(cfg.get("seed", 42))
    name, modality = cfg["name"], cfg["modality"]
    print(f"{'='*60}\n{name}\n{'='*60}")

    if USE_WANDB:
        wandb.init(project=WANDB_PROJECT, name=name,
                   group=modality, config={**cfg, "emotions": EMOTIONS}, reinit=True)

    train_ds = EmotionDataset(METADATA, "train", modality)
    val_ds = EmotionDataset(METADATA, "val", modality)
    bs = cfg.get("batch_size", 8)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=0, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False,
                            num_workers=0, collate_fn=collate_fn)

    model, processor, prep_fn = build_model_and_prep(cfg)
    model.to(DEVICE)
    loss_fn = build_loss_fn(cfg)

    wd = cfg.get("weight_decay", DEFAULT_WEIGHT_DECAY)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"], weight_decay=wd)
    scaler = GradScaler(enabled=DEVICE == "cuda")
    scheduler = None

    best_f1, patience_cnt = 0.0, 0
    save_path = OUT_DIR / name

    for epoch in range(cfg["epochs"]):
        freeze_ep = cfg.get("freeze_epochs", 2)
        if epoch == freeze_ep:
            for p in model.parameters():
                p.requires_grad = True
            unfreeze_lr = cfg["lr"] if freeze_ep == 0 else cfg["lr"] * 0.1
            optimizer = torch.optim.AdamW(model.parameters(), lr=unfreeze_lr, weight_decay=wd)
            scaler = GradScaler(enabled=DEVICE == "cuda")
            scheduler = CosineAnnealingLR(
                optimizer, T_max=cfg["epochs"] - epoch, eta_min=1e-7)

        t = train_one_epoch(model, train_loader, prep_fn, optimizer, scaler, loss_fn)
        v = evaluate(model, val_loader, prep_fn, loss_fn)

        if scheduler:
            scheduler.step()

        if USE_WANDB:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": t["loss"], "train/acc": t["acc"], "train/f1": t["f1"],
                "val/loss": v["loss"], "val/acc": v["acc"],
                "val/f1": v["f1"], "val/f1_macro": v["f1_macro"],
                "lr": optimizer.param_groups[0]["lr"],
            })
        print(f"  [{epoch+1:2d}/{cfg['epochs']}] "
              f"t_f1={t['f1']:.3f} v_f1={v['f1']:.3f} v_f1m={v['f1_macro']:.3f} v_loss={v['loss']:.3f}")

        if v["f1"] > best_f1:
            best_f1 = v["f1"]
            save_path.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(save_path))
            processor.save_pretrained(str(save_path))
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.get("patience", 5):
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if USE_WANDB:
        wandb.log({"best_val_f1": best_f1})
        wandb.finish()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f"  Best F1: {best_f1:.4f} -> {save_path}\n")
    return {"name": name, "best_f1": best_f1, "path": str(save_path), "modality": modality}

## Experiment grid (expanded for 6-class task)

Extra axes vs. 3-emotion notebook:
- **label_smoothing** (0.0 vs 0.1) — tests whether LS helps with more classes
- **weight_decay** (0.01 vs 0.05) — stronger regularization for harder task
- **class_balanced** loss — counteracts any residual class imbalance
- **freeze_epochs=0** — full fine-tuning from epoch 0
- **extra frame counts** (4, 12) and **window sizes** (1.5s) for video/audio

In [8]:
P = "6emo-"

AUDIO_EXPERIMENTS = [
    # wav2vec2-base-er: learning-rate sweep
    {"name": f"{P}w2v2-er-lr3e5", "modality": "audio",
     "model": "superb/wav2vec2-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}w2v2-er-lr5e5", "modality": "audio",
     "model": "superb/wav2vec2-base-superb-er",
     "lr": 5e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}w2v2-er-lr1e4", "modality": "audio",
     "model": "superb/wav2vec2-base-superb-er",
     "lr": 1e-4, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    # wav2vec2-base-er: window-length sweep
    {"name": f"{P}w2v2-er-lr5e5-w5", "modality": "audio",
     "model": "superb/wav2vec2-base-superb-er",
     "lr": 5e-5, "window_s": 5.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}w2v2-er-lr5e5-w15", "modality": "audio",
     "model": "superb/wav2vec2-base-superb-er",
     "lr": 5e-5, "window_s": 1.5, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},

    # hubert-base-er: learning-rate sweep
    {"name": f"{P}hubert-er-lr3e5", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}hubert-er-lr5e5", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 5e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}hubert-er-lr1e4", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 1e-4, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    {"name": f"{P}hubert-er-lr5e5-w5", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 5e-5, "window_s": 5.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8},
    # hubert-base-er: regularization/freeze ablations
    {"name": f"{P}hubert-er-lr3e5-nf", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 0, "patience": 8},
    {"name": f"{P}hubert-er-lr3e5-ls0", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8,
     "label_smoothing": 0.0},
    {"name": f"{P}hubert-er-lr3e5-cb", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 2, "patience": 8,
     "class_balanced": True},

    # large backbones
    {"name": f"{P}w2v2-lg-lr2e5", "modality": "audio",
     "model": "facebook/wav2vec2-large",
     "lr": 2e-5, "window_s": 3.0, "batch_size": 4,
     "epochs": 40, "freeze_epochs": 3, "patience": 7},
    {"name": f"{P}hubert-lg-lr2e5", "modality": "audio",
     "model": "facebook/hubert-large-ll60k",
     "lr": 2e-5, "window_s": 3.0, "batch_size": 4,
     "epochs": 40, "freeze_epochs": 3, "patience": 7},
    {"name": f"{P}hubert-lg-lr1e5-wd05", "modality": "audio",
     "model": "facebook/hubert-large-ll60k",
     "lr": 1e-5, "window_s": 3.0, "batch_size": 4,
     "epochs": 40, "freeze_epochs": 3, "patience": 7,
     "weight_decay": 0.05},
]

VIDEO_EXPERIMENTS = [
    # timesformer 16-frame: lr sweep
    {"name": f"{P}tsf-lr1e5-16f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 1e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    {"name": f"{P}tsf-lr3e5-16f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    {"name": f"{P}tsf-lr5e5-16f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 5e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    # freeze-schedule ablations
    {"name": f"{P}tsf-lr3e5-16f-nf", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 0, "patience": 7},
    {"name": f"{P}tsf-lr1e5-16f-f3", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 1e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 3, "patience": 7},
    # frame-count sweep
    {"name": f"{P}tsf-lr3e5-4f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 4, "batch_size": 4,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    {"name": f"{P}tsf-lr3e5-8f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 8, "batch_size": 4,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    {"name": f"{P}tsf-lr3e5-12f", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 12, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7},
    # regularization ablations
    {"name": f"{P}tsf-lr3e5-16f-ls0", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7,
     "label_smoothing": 0.0},
    {"name": f"{P}tsf-lr3e5-16f-wd05", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7,
     "weight_decay": 0.05},
    {"name": f"{P}tsf-lr3e5-16f-cb", "modality": "video",
     "model": "facebook/timesformer-base-finetuned-k400",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 30, "freeze_epochs": 1, "patience": 7,
     "class_balanced": True},
]

EXPERIMENTS = AUDIO_EXPERIMENTS + VIDEO_EXPERIMENTS
print(f"Total experiments: {len(EXPERIMENTS)} "
      f"({len(AUDIO_EXPERIMENTS)} audio, {len(VIDEO_EXPERIMENTS)} video)")

Total experiments: 26 (15 audio, 11 video)


In [9]:
results = []
for exp in EXPERIMENTS:
    results.append(run_experiment(exp))

6emo-w2v2-er-lr3e5


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at superb/wav2vec2-base-superb-er and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([4, 256]) in the checkpoint and torch.Size([6, 256]) in the model instantiated
- classifier.bias: found shape torch.Size([4]) in the checkpoint and torch.Size([6]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

  [20/30] t_f1=1.000 v_f1=0.573 v_f1m=0.573 v_loss=1.400


  [21/30] t_f1=1.000 v_f1=0.578 v_f1m=0.578 v_loss=1.385
  Early stopping at epoch 21


best_val_f1,▁
epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
lr,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▄▆▇▇████████████████
train/f1,▁▄▆▇▇████████████████
train/loss,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▅▇▅▆▇▇▇█▇▆▇▆█▇█▇▇▇▇▇
val/f1,▁▅▇▅▇▇███▇▇▇▇█▇██▇▇▇█
val/f1_macro,▁▅▇▅▇▇███▇▇▇▇█▇██▇▇▇█
val/loss,█▃▃▆▃▂▁▂▂▂▃▅▄▂▄▂▂▃▃▃▃
best_val_f1,0.59966


  Best F1: 0.5997 -> /content/trained_encoders_6emotions/6emo-tsf-lr5e5-16f

6emo-tsf-lr3e5-16f-nf


CommError: Run initialization has timed out after 90.0 sec. Please try increasing the timeout with the `init_timeout` setting: `wandb.init(settings=wandb.Settings(init_timeout=120))`.

In [10]:
print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}")
print(f"{'Name':40s} {'Modality':>8s} {'Best Val F1':>12s}")
print("-" * 63)
for r in sorted(results, key=lambda x: -x["best_f1"]):
    print(f"{r['name']:40s} {r['modality']:>8s} {r['best_f1']:12.4f}")

best_audio = max((r for r in results if r["modality"] == "audio"), key=lambda x: x["best_f1"])
best_video = max((r for r in results if r["modality"] == "video"), key=lambda x: x["best_f1"])
print(f"\nBest audio: {best_audio['name']} (F1={best_audio['best_f1']:.4f})")
print(f"Best video: {best_video['name']} (F1={best_video['best_f1']:.4f})")


RESULTS SUMMARY
Name                                     Modality  Best Val F1
---------------------------------------------------------------
6emo-hubert-er-lr3e5-nf                     audio       0.7816
6emo-w2v2-er-lr5e5-w5                       audio       0.7339
6emo-w2v2-er-lr1e4                          audio       0.7233
6emo-hubert-er-lr1e4                        audio       0.7070
6emo-hubert-lg-lr2e5                        audio       0.6959
6emo-w2v2-er-lr3e5                          audio       0.6902
6emo-hubert-er-lr5e5                        audio       0.6848
6emo-w2v2-er-lr5e5                          audio       0.6814
6emo-hubert-er-lr5e5-w5                     audio       0.6748
6emo-hubert-er-lr3e5                        audio       0.6402
6emo-hubert-er-lr3e5-cb                     audio       0.6402
6emo-tsf-lr3e5-16f                          video       0.6124
6emo-w2v2-er-lr5e5-w15                      audio       0.6108
6emo-tsf-lr5e5-16f                   

In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [12]:
@torch.no_grad()
def collect_predictions(model, loader, prep_fn):
    model.eval()
    all_preds, all_labels, all_losses = [], [], []
    for batch in loader:
        kwargs, y = prep_fn(batch, train=False)
        with autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(**kwargs).logits
            loss = nn.CrossEntropyLoss(reduction="none")(logits, y)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.cpu().tolist())
        all_losses.extend(loss.cpu().tolist())
    return np.array(all_preds), np.array(all_labels), np.array(all_losses)


def per_emotion_report(preds, labels, losses, title):
    import matplotlib.pyplot as plt
    import seaborn as sns

    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}\n")
    print(classification_report(labels, preds, target_names=EMOTIONS, digits=3, zero_division=0))

    per_f1 = f1_score(labels, preds, average=None,
                      labels=list(range(NUM_EMOTIONS)), zero_division=0)
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_EMOTIONS)))

    print(f"{'Emotion':<12s} {'N':>5s} {'Acc':>6s} {'F1':>6s} {'AvgLoss':>8s}")
    print("-" * 40)
    for i, emo in enumerate(EMOTIONS):
        mask = labels == i
        n = mask.sum()
        if n == 0:
            continue
        acc = (preds[mask] == i).mean()
        avg_loss = losses[mask].mean()
        print(f"{emo:<12s} {n:5d} {acc:6.3f} {per_f1[i]:6.3f} {avg_loss:8.3f}")

    worst = np.argsort(per_f1)
    print(f"\nWeakest: ", end="")
    print(" < ".join(f"{EMOTIONS[i]} ({per_f1[i]:.3f})" for i in worst[:3]))

    fig, ax = plt.subplots(figsize=(9, 7))
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm / row_sums
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax,
                vmin=0, vmax=1, linewidths=0.5)
    for i in range(NUM_EMOTIONS):
        for j in range(NUM_EMOTIONS):
            if cm[i, j] > 0:
                ax.text(j + 0.5, i + 0.72, f"({cm[i,j]})",
                        ha="center", va="center", fontsize=7, color="gray")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return per_f1

## Patch — AffectNet-ViT per-frame baseline + focal loss

Two targeted fixes for the weak spots in the first round:

- **Audio** (`fearful` recall ≈ 0.50): add focal-loss and per-class weighting.
- **Video** (`surprised` F1 ≈ 0.15 — TimeSformer Kinetics-400 prior has no emotion signal):
  add an **AffectNet-pretrained ViT** (`trpakov/vit-face-expression`) applied per-frame
  with temporal mean-pooling over logits.

New config keys: `focal_gamma`, `class_weights`, `video_arch="perframe_vit"`.


In [ ]:
import torch.nn.functional as F
from types import SimpleNamespace
from transformers import AutoModelForImageClassification


class FocalLoss(nn.Module):
    """Focal CE with optional per-class weighting and label smoothing."""
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = float(gamma)
        self.weight = weight  # tensor or None
        self.label_smoothing = float(label_smoothing)

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        pt = p.gather(1, target.unsqueeze(1)).squeeze(1).clamp(min=1e-8)
        focal = (1.0 - pt).pow(self.gamma)
        ce = F.cross_entropy(
            logits, target,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        return (focal * ce).mean()


class PerFrameViTEmotion(nn.Module):
    """Wrap a per-image ViT classifier; mean-pool logits over T frames.

    Input  : pixel_values [B*T, C, H, W], n_frames=T
    Output : SimpleNamespace(logits=[B, num_labels])
    """
    _marker_name = "perframe_vit.json"

    def __init__(self, base_model_name, num_labels=NUM_EMOTIONS):
        super().__init__()
        self.backbone = AutoModelForImageClassification.from_pretrained(
            base_model_name,
            num_labels=num_labels,
            ignore_mismatched_sizes=True,
        )
        self.base_model_name = base_model_name

    def forward(self, pixel_values, n_frames, **kwargs):
        out = self.backbone(pixel_values=pixel_values)
        logits = out.logits  # [B*T, C]
        BT, C = logits.shape
        B = BT // int(n_frames)
        logits = logits.view(B, int(n_frames), C).mean(dim=1)
        return SimpleNamespace(logits=logits)

    def save_pretrained(self, path):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        self.backbone.save_pretrained(str(path))
        with open(path / self._marker_name, "w") as fh:
            json.dump({"arch": "perframe_vit",
                       "base_model_name": self.base_model_name}, fh)

    @classmethod
    def from_pretrained(cls, path, num_labels=NUM_EMOTIONS):
        inst = cls.__new__(cls)
        nn.Module.__init__(inst)
        inst.backbone = AutoModelForImageClassification.from_pretrained(
            str(path), num_labels=num_labels, ignore_mismatched_sizes=True,
        )
        marker = Path(path) / cls._marker_name
        if marker.exists():
            with open(marker) as fh:
                inst.base_model_name = json.load(fh).get("base_model_name", "")
        else:
            inst.base_model_name = ""
        return inst


def prepare_video_perframe(batch, processor, n_frames, device, train=True):
    """Flat per-frame preprocessing for per-image ViT classifiers."""
    flat_frames = []
    for v in batch["video"]:
        clip = crop_video(v, n_frames, train)  # [T, C, H, W]
        for i in range(clip.shape[0]):
            flat_frames.append(clip[i].permute(1, 2, 0).numpy())
    enc = processor(images=flat_frames, return_tensors="pt", do_rescale=False)
    return (
        {"pixel_values": enc["pixel_values"].to(device),
         "n_frames": n_frames},
        batch["emotion"].to(device),
    )


# Preserve originals, then upgrade
_orig_build_loss_fn = build_loss_fn
_orig_build_model_and_prep = build_model_and_prep


def build_loss_fn_v2(cfg):
    ls = cfg.get("label_smoothing", DEFAULT_LABEL_SMOOTHING)
    weight = None
    if cfg.get("class_balanced", False):
        counts = class_counts(METADATA, "train").astype(np.float32)
        inv = counts.sum() / (NUM_EMOTIONS * np.clip(counts, 1, None))
        weight = torch.tensor(inv, dtype=torch.float32, device=DEVICE)
    elif cfg.get("class_weights") is not None:
        weight = torch.tensor(cfg["class_weights"], dtype=torch.float32, device=DEVICE)
        assert weight.numel() == NUM_EMOTIONS, \
            f"class_weights len {weight.numel()} != {NUM_EMOTIONS}"
    gamma = cfg.get("focal_gamma", 0.0) or 0.0
    if gamma > 0:
        return FocalLoss(gamma=gamma, weight=weight, label_smoothing=ls)
    return nn.CrossEntropyLoss(label_smoothing=ls, weight=weight)


def build_model_and_prep_v2(cfg):
    if cfg["modality"] != "video" or cfg.get("video_arch", "timesformer") != "perframe_vit":
        return _orig_build_model_and_prep(cfg)
    model = PerFrameViTEmotion(cfg["model"], num_labels=NUM_EMOTIONS)
    processor = AutoImageProcessor.from_pretrained(cfg["model"])
    prep_fn = partial(prepare_video_perframe, processor=processor,
                      n_frames=cfg.get("n_frames", 8), device=DEVICE)
    for n, p in model.named_parameters():
        if "classifier" not in n:
            p.requires_grad = False
    return model, processor, prep_fn


# Install — run_experiment looks these up at call-time
build_loss_fn = build_loss_fn_v2
build_model_and_prep = build_model_and_prep_v2
print("Patched: build_loss_fn, build_model_and_prep (v2 with focal / perframe_vit)")


In [ ]:
# Audio: target fearful recall (0.50) with focal-loss and per-class weighting.
# class_weights order = EMOTIONS = [happy, sad, angry, fearful, disgust, surprised]
NEW_AUDIO_EXPERIMENTS = [
    {"name": f"{P}hubert-er-lr3e5-nf-focal", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 0, "patience": 8,
     "focal_gamma": 2.0},
    {"name": f"{P}hubert-er-lr3e5-nf-cw", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 0, "patience": 8,
     "class_weights": [1.0, 1.2, 1.0, 1.8, 1.0, 1.2]},
    {"name": f"{P}hubert-er-lr3e5-nf-focal-cw", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 3.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 0, "patience": 8,
     "focal_gamma": 2.0,
     "class_weights": [1.0, 1.2, 1.0, 1.8, 1.0, 1.2]},
    {"name": f"{P}hubert-er-lr3e5-nf-w5-focal", "modality": "audio",
     "model": "superb/hubert-base-superb-er",
     "lr": 3e-5, "window_s": 5.0, "batch_size": 8,
     "epochs": 50, "freeze_epochs": 0, "patience": 8,
     "focal_gamma": 2.0},
]

# Video: AffectNet-pretrained ViT per-frame, mean-pooled over T. Strong emotion prior.
NEW_VIDEO_EXPERIMENTS = [
    {"name": f"{P}affnet-vit-8f-frozen", "modality": "video",
     "model": "trpakov/vit-face-expression", "video_arch": "perframe_vit",
     "lr": 1e-4, "n_frames": 8, "batch_size": 4,
     "epochs": 25, "freeze_epochs": 999, "patience": 7},  # head-only
    {"name": f"{P}affnet-vit-8f-lr3e5", "modality": "video",
     "model": "trpakov/vit-face-expression", "video_arch": "perframe_vit",
     "lr": 3e-5, "n_frames": 8, "batch_size": 4,
     "epochs": 25, "freeze_epochs": 2, "patience": 7},
    {"name": f"{P}affnet-vit-16f-lr3e5", "modality": "video",
     "model": "trpakov/vit-face-expression", "video_arch": "perframe_vit",
     "lr": 3e-5, "n_frames": 16, "batch_size": 2,
     "epochs": 25, "freeze_epochs": 2, "patience": 7},
    {"name": f"{P}affnet-vit-8f-focal-cw", "modality": "video",
     "model": "trpakov/vit-face-expression", "video_arch": "perframe_vit",
     "lr": 3e-5, "n_frames": 8, "batch_size": 4,
     "epochs": 25, "freeze_epochs": 2, "patience": 7,
     "focal_gamma": 2.0,
     "class_weights": [1.0, 1.2, 1.0, 1.2, 1.0, 1.5]},
]

NEW_EXPERIMENTS = NEW_AUDIO_EXPERIMENTS + NEW_VIDEO_EXPERIMENTS
print(f"New experiments: {len(NEW_EXPERIMENTS)} "
      f"({len(NEW_AUDIO_EXPERIMENTS)} audio, {len(NEW_VIDEO_EXPERIMENTS)} video)")


In [ ]:
for exp in NEW_EXPERIMENTS:
    results.append(run_experiment(exp))
    EXPERIMENTS.append(exp)  # so downstream cells can look up cfg by name


In [ ]:
print(f"\n{'='*60}")
print("RESULTS SUMMARY (all runs, incl. new)")
print(f"{'='*60}")
print(f"{'Name':45s} {'Modality':>8s} {'Best Val F1':>12s}")
print("-" * 68)
for r in sorted(results, key=lambda x: -x["best_f1"]):
    print(f"{r['name']:45s} {r['modality']:>8s} {r['best_f1']:12.4f}")

best_audio = max((r for r in results if r["modality"] == "audio"), key=lambda x: x["best_f1"])
best_video = max((r for r in results if r["modality"] == "video"), key=lambda x: x["best_f1"])
print(f"\nBest audio overall: {best_audio['name']}  F1={best_audio['best_f1']:.4f}")
print(f"Best video overall: {best_video['name']}  F1={best_video['best_f1']:.4f}")


In [ ]:
import matplotlib.pyplot as plt
# Re-evaluate the refreshed best_audio / best_video with per-emotion breakdown.
# The video loader branches on `video_arch == "perframe_vit"` so the ViT path works.

def _load_audio_model(path, name):
    cls = (HubertForSequenceClassification if "hubert" in name.lower()
           else Wav2Vec2ForSequenceClassification)
    return cls.from_pretrained(path).to(DEVICE)


def _load_video_model(path, cfg):
    if cfg.get("video_arch") == "perframe_vit":
        return PerFrameViTEmotion.from_pretrained(path).to(DEVICE)
    return TimesformerForVideoClassification.from_pretrained(path).to(DEVICE)


# --- Audio ---
audio_cfg = next(e for e in EXPERIMENTS if e["name"] == best_audio["name"])
audio_model = _load_audio_model(best_audio["path"], best_audio["name"])
audio_processor = Wav2Vec2FeatureExtractor.from_pretrained(best_audio["path"])
audio_prep = partial(prepare_audio, processor=audio_processor,
                     window_s=audio_cfg.get("window_s", 3.0), device=DEVICE)

val_audio = EmotionDataset(METADATA, "val", "audio")
audio_loader = DataLoader(val_audio, batch_size=8, shuffle=False, collate_fn=collate_fn)
print(f"Evaluating audio: {best_audio['name']} (F1={best_audio['best_f1']:.4f})")
a_preds, a_labels, a_losses = collect_predictions(audio_model, audio_loader, audio_prep)
a_f1 = per_emotion_report(a_preds, a_labels, a_losses, f"AUDIO — {best_audio['name']}")
del audio_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Video ---
video_cfg = next(e for e in EXPERIMENTS if e["name"] == best_video["name"])
video_model = _load_video_model(best_video["path"], video_cfg)
video_processor = AutoImageProcessor.from_pretrained(best_video["path"])
if video_cfg.get("video_arch") == "perframe_vit":
    video_prep = partial(prepare_video_perframe, processor=video_processor,
                         n_frames=video_cfg.get("n_frames", 8), device=DEVICE)
    vbs = 4 if video_cfg.get("n_frames", 8) <= 8 else 2
else:
    video_prep = partial(prepare_video, processor=video_processor,
                         n_frames=video_cfg.get("n_frames", 16), device=DEVICE)
    vbs = 2

val_video = EmotionDataset(METADATA, "val", "video")
video_loader = DataLoader(val_video, batch_size=vbs, shuffle=False, collate_fn=collate_fn)
print(f"Evaluating video: {best_video['name']} (F1={best_video['best_f1']:.4f})")
v_preds, v_labels, v_losses = collect_predictions(video_model, video_loader, video_prep)
v_f1 = per_emotion_report(v_preds, v_labels, v_losses, f"VIDEO — {best_video['name']}")
del video_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Combined per-emotion bar chart ---
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(NUM_EMOTIONS); w = 0.35
ax.bar(x - w/2, a_f1, w, label=f"Audio ({best_audio['name']})", color="#4C72B0")
ax.bar(x + w/2, v_f1, w, label=f"Video ({best_video['name']})", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(EMOTIONS, rotation=30, ha="right")
ax.set_ylabel("F1 Score"); ax.set_ylim(0, 1)
ax.set_title(f"Per-Emotion F1 — refreshed bests ({NUM_EMOTIONS} classes)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i in x:
    ax.text(i - w/2, a_f1[i] + 0.02, f"{a_f1[i]:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, v_f1[i] + 0.02, f"{v_f1[i]:.2f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()


print(f"\n{'='*60}")
print("CROSS-MODAL GAPS")
print(f"{'='*60}")
print(f"{'Emotion':<12s} {'Audio F1':>9s} {'Video F1':>9s} {'Gap':>7s}")
print("-" * 42)
for i, emo in enumerate(EMOTIONS):
    gap = abs(a_f1[i] - v_f1[i])
    better = "A" if a_f1[i] > v_f1[i] else "V"
    print(f"{emo:<12s} {a_f1[i]:9.3f} {v_f1[i]:9.3f} {gap:7.3f} ({better})")

overall_a = f1_score(a_labels, a_preds, average="weighted")
overall_v = f1_score(v_labels, v_preds, average="weighted")
print(f"\nOverall weighted F1 — Audio: {overall_a:.4f}, Video: {overall_v:.4f}")
